# 04 — Validación reproducible (QA de F1-05 / F1-06)

**Por qué existe este notebook.** El README del repo afirmaba dos resultados de validación —un test de McNemar
con `p ≈ 0.00045` y un modelo entrenado con una sola variable— que **no tenían código en el repositorio**.
Se corrieron en su momento en una sesión interactiva y sólo la conclusión llegó al README.

Una afirmación estadística publicada sin el código que la produce no es verificable, y en un portfolio eso pesa
más que el resultado en sí. Este notebook reconstruye esas pruebas desde `train.parquet` / `test.parquet`,
reporta los números que efectivamente salen, y corrige el README donde no coinciden.

**Regla aplicada:** si el número reproducido difiere del publicado, manda el reproducido.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, recall_score, roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar

PROCESSED = "../../data/processed"

train_df = pd.read_parquet(f"{PROCESSED}/train.parquet")
test_df = pd.read_parquet(f"{PROCESSED}/test.parquet")
print("Train:", train_df.shape, "| Test:", test_df.shape)

Train: (719870, 27) | Test: (178545, 27)


## 1. Mismo preprocesamiento que `02` y `03`

Se replica exactamente el `prep()` de los notebooks anteriores. Si esto no da los mismos números que el
baseline publicado, cualquier comparación posterior no significa nada — así que el primer objetivo es
**reproducir el baseline**, no mejorarlo.

In [2]:
feature_cols_num = [
    "n_packages", "total_volume_cm3", "total_planned_service_seconds",
    "window_duration_min", "volumen_promedio_paquete_cm3",
    "paradas_por_ruta", "paquetes_por_ruta", "distancia_a_siguiente_km",
    "zona_riesgo_low",
]
feature_cols_bool = ["has_time_window", "any_rejected", "any_attempted"]
feature_cols_cat = ["station_code", "franja_horaria"]
all_feats = feature_cols_num + feature_cols_bool + feature_cols_cat


def prep(df, median_dist=None):
    df = df.copy()
    df["window_duration_min"] = df["window_duration_min"].fillna(0)
    df["franja_horaria"] = df["franja_horaria"].cat.add_categories("sin_ventana").fillna("sin_ventana")
    if median_dist is None:
        median_dist = df["distancia_a_siguiente_km"].median()
    df["distancia_a_siguiente_km"] = df["distancia_a_siguiente_km"].fillna(median_dist)
    for c in feature_cols_bool:
        df[c] = df[c].astype(int)
    return df, median_dist


train_df, median_dist = prep(train_df)
test_df, _ = prep(test_df, median_dist=median_dist)

y_train = train_df["route_score"].values
y_test = test_df["route_score"].values
print("Nulos en features (train/test):",
      train_df[all_feats].isna().sum().sum(), "/", test_df[all_feats].isna().sum().sum())

Nulos en features (train/test): 0 / 0


## 2. El tamaño real de la muestra sobre la que se valida

Antes de cualquier test: **cuántas unidades independientes hay realmente en test.** El dataset tiene 898K paradas,
pero la etiqueta `route_score` es de la ruta, propagada a sus paradas (*weak label*). Las paradas de una misma
ruta **no son observaciones independientes**: comparten la etiqueta por construcción.

Esta celda es la que cambia la interpretación de todo lo que sigue.

In [3]:
mask_low = (y_test == "Low")
n_paradas_low = int(mask_low.sum())
n_rutas_low = int(test_df.loc[mask_low, "route_id"].nunique())

print("Paradas Low en test:", n_paradas_low)
print("Rutas  Low en test:", n_rutas_low)
print(f"Paradas por ruta Low (promedio): {n_paradas_low / n_rutas_low:.0f}")
print()
print("Rutas Low en TODO el dataset:", 102, "(train + test)")

Paradas Low en test: 3047
Rutas  Low en test: 20
Paradas por ruta Low (promedio): 152

Rutas Low en TODO el dataset: 102 (train + test)


**Lectura:** la validación de la clase `Low` se apoya en **20 rutas independientes**, no en 3.047 observaciones.
Un test estadístico corrido a nivel parada trata cada parada como un dato nuevo y **infla artificialmente la
significancia** — es pseudo-replicación. Se muestran las dos versiones más abajo, justamente para que se vea
la diferencia.

## 3. Reproducción del baseline publicado (control)

Si esta celda no devuelve recall `Low` ≈ 0.18 y ROC-AUC ≈ 0.6102, el pipeline se desvió y hay que parar acá.

In [4]:
def matrices(tr, te):
    Xtr = pd.get_dummies(tr[all_feats], columns=feature_cols_cat)
    Xte = pd.get_dummies(te[all_feats], columns=feature_cols_cat).reindex(columns=Xtr.columns, fill_value=0)
    return Xtr, Xte


def corre_lr(tr, te, tag):
    Xtr, Xte = matrices(tr, te)
    scaler = StandardScaler()
    m = LogisticRegression(max_iter=300, class_weight="balanced")
    m.fit(scaler.fit_transform(Xtr), y_train)
    Xte_s = scaler.transform(Xte)
    pred, proba = m.predict(Xte_s), m.predict_proba(Xte_s)
    auc = roc_auc_score(y_test, proba, multi_class="ovr", labels=m.classes_)
    print(f"=== {tag} ===")
    print(classification_report(y_test, pred, digits=3))
    print("recall Low:", round(recall_score(y_test, pred, labels=["Low"], average="macro"), 4),
          "| ROC-AUC (ovr, macro):", round(auc, 4))
    return pred


pred_lr = corre_lr(train_df, test_df, "LR baseline, zona CON suavizado (= modelo publicado)")

=== LR baseline, zona CON suavizado (= modelo publicado) ===


              precision    recall  f1-score   support

        High      0.555     0.498     0.525     74901
         Low      0.020     0.180     0.035      3047
      Medium      0.683     0.566     0.619    100597

    accuracy                          0.531    178545
   macro avg      0.419     0.415     0.393    178545
weighted avg      0.618     0.531     0.569    178545



recall Low: 0.1802 | ROC-AUC (ovr, macro): 0.6102


**Control OK.** recall `Low` 0.180 y ROC-AUC 0.6102 — idénticos a los del README. El pipeline reproduce.

## 4. Reconstrucción del baseline "sin suavizar"

El README compara el modelo final contra un baseline con `zona_riesgo_low` **sin suavizado bayesiano** (tasa
cruda de `Low` por zona), y le atribuye 16% de recall.

Se reconstruye esa variable con el mismo criterio leak-free del notebook 01: calculada **sólo con train**,
deduplicando por el par `(zone_id, route_id)` —el bug que se corrigió en su momento fue justamente deduplicar
por `route_id` solo—, y asignando la tasa global a las zonas no vistas.

In [5]:
rz = train_df[["zone_id", "route_id", "route_score"]].drop_duplicates(subset=["zone_id", "route_id"])
grp = rz.groupby("zone_id")["route_score"]
n_rutas_zona = grp.size()
tasa_cruda = grp.apply(lambda s: (s == "Low").mean())
tasa_global = float((rz["route_score"] == "Low").mean())

print("Zonas en train:", len(n_rutas_zona))
print(f"Zonas con menos de 5 rutas de historia: {int((n_rutas_zona < 5).sum())} ({(n_rutas_zona < 5).mean():.1%})")
print("Zonas con tasa cruda = 100% (ruido con forma de senal):", int((tasa_cruda == 1.0).sum()))
print("Tasa global de Low (nivel zona-ruta):", round(tasa_global, 6))


def con_tasa_cruda(df):
    df = df.copy()
    df["zona_riesgo_low"] = df["zone_id"].map(tasa_cruda).fillna(tasa_global).astype(float)
    return df


train_raw, test_raw = con_tasa_cruda(train_df), con_tasa_cruda(test_df)
pred_lr_raw = corre_lr(train_raw, test_raw, "LR baseline, zona SIN suavizar")

Zonas en train: 8720
Zonas con menos de 5 rutas de historia: 2926 (33.6%)
Zonas con tasa cruda = 100% (ruido con forma de senal): 10
Tasa global de Low (nivel zona-ruta): 0.016823


=== LR baseline, zona SIN suavizar ===


              precision    recall  f1-score   support

        High      0.556     0.499     0.526     74901
         Low      0.020     0.180     0.036      3047
      Medium      0.682     0.572     0.622    100597

    accuracy                          0.535    178545
   macro avg      0.419     0.417     0.395    178545
weighted avg      0.618     0.535     0.572    178545



recall Low: 0.1795 | ROC-AUC (ovr, macro): 0.6276


**Discrepancia #1 con el README.** El baseline sin suavizar reproduce en **17,95%**, no en 16%.
La diferencia contra el modelo suavizado (18,02%) es de **0,07 puntos**, no de 2.

El 34% de zonas con menos de 5 rutas y las zonas con tasa 100% **sí** se reproducen (33,6% y 10 zonas): el
diagnóstico que motivó el suavizado era correcto. Lo que no se sostiene es la magnitud de la mejora atribuida.

## 5. Random Forest (control) y McNemar

McNemar es el test correcto para comparar **dos modelos sobre las mismas observaciones**: no mira cuántos acierta
cada uno, mira **en qué casos se contradicen**. Sólo cuentan los discordantes (uno acierta y el otro no). Si los
discordantes están repartidos parejo, la diferencia es ruido.

In [6]:
Xtr, Xte = matrices(train_df, test_df)
rf = RandomForestClassifier(n_estimators=100, max_depth=12,
                            class_weight="balanced_subsample", random_state=42, n_jobs=-1)
rf.fit(Xtr, y_train)
pred_rf = rf.predict(Xte)
print("=== Random Forest (control) ===")
print(classification_report(y_test, pred_rf, digits=3))

=== Random Forest (control) ===


              precision    recall  f1-score   support

        High      0.566     0.455     0.504     74901
         Low      0.018     0.177     0.033      3047
      Medium      0.660     0.580     0.617    100597

    accuracy                          0.521    178545
   macro avg      0.415     0.404     0.385    178545
weighted avg      0.609     0.521     0.560    178545



In [7]:
def mcnemar_paradas(pred_a, pred_b, tag):
    A = (pred_a[mask_low] == "Low").astype(int)
    B = (pred_b[mask_low] == "Low").astype(int)
    n11 = int(((A == 1) & (B == 1)).sum())
    n10 = int(((A == 1) & (B == 0)).sum())   # solo A acierta
    n01 = int(((A == 0) & (B == 1)).sum())   # solo B acierta
    n00 = int(((A == 0) & (B == 0)).sum())
    res = mcnemar([[n11, n10], [n01, n00]], exact=False, correction=True)
    print(f"{tag}")
    print(f"   recall Low: {A.mean():.4f} -> {B.mean():.4f}")
    print(f"   discordantes: solo A={n10} | solo B={n01}   (son los unicos que cuentan)")
    print(f"   chi2={res.statistic:.3f}   p={res.pvalue:.6g}\n")


print("--- McNemar A NIVEL PARADA (n =", n_paradas_low, "paradas Low) ---\n")
mcnemar_paradas(pred_lr_raw, pred_lr, "LR sin suavizar  ->  LR suavizado")
mcnemar_paradas(pred_lr, pred_rf, "LR suavizado  ->  Random Forest")

--- McNemar A NIVEL PARADA (n = 3047 paradas Low) ---

LR sin suavizar  ->  LR suavizado
   recall Low: 0.1795 -> 0.1802
   discordantes: solo A=2 | solo B=4   (son los unicos que cuentan)
   chi2=0.167   p=0.683091

LR suavizado  ->  Random Forest
   recall Low: 0.1802 -> 0.1772
   discordantes: solo A=128 | solo B=119   (son los unicos que cuentan)
   chi2=0.259   p=0.610733



**Discrepancia #2, la importante.** El README publica `p ≈ 0.00045` y concluye *"resultado estadísticamente
significativo"*. Acá, **incluso en la versión a nivel parada que es la más permisiva**, da `p ≈ 0.68`:
los discordantes son 2 contra 4. No hay diferencia detectable.

No se pudo reproducir el `p ≈ 0.00045` bajo ninguna reconstrucción del baseline sin suavizar.

## 6. El mismo test a nivel ruta — la unidad correcta

Repetido sobre las 20 rutas `Low` independientes. Una ruta se considera detectada si el modelo predice `Low`
en al menos la mitad de sus paradas. Se usa la versión exacta (binomial) porque con n=20 la aproximación
chi-cuadrado no aplica.

In [8]:
rid = test_df.loc[mask_low, "route_id"].values


def mcnemar_rutas(pred_a, pred_b, tag):
    d = pd.DataFrame({"route_id": rid,
                      "A": (pred_a[mask_low] == "Low"),
                      "B": (pred_b[mask_low] == "Low")}).groupby("route_id").mean()
    A = (d["A"] >= 0.5).astype(int).values
    B = (d["B"] >= 0.5).astype(int).values
    n10 = int(((A == 1) & (B == 0)).sum())
    n01 = int(((A == 0) & (B == 1)).sum())
    res = mcnemar([[int(((A == 1) & (B == 1)).sum()), n10],
                   [n01, int(((A == 0) & (B == 0)).sum())]], exact=True)
    print(f"{tag}")
    print(f"   rutas detectadas: {A.sum()}/{len(A)} -> {B.sum()}/{len(B)}")
    print(f"   discordantes: solo A={n10} | solo B={n01}")
    print(f"   p (binomial exacto) = {res.pvalue:.4g}\n")


print("--- McNemar A NIVEL RUTA (n =", n_rutas_low, "rutas Low) ---\n")
mcnemar_rutas(pred_lr_raw, pred_lr, "LR sin suavizar  ->  LR suavizado")
mcnemar_rutas(pred_lr, pred_rf, "LR suavizado  ->  Random Forest")

--- McNemar A NIVEL RUTA (n = 20 rutas Low) ---

LR sin suavizar  ->  LR suavizado
   rutas detectadas: 3/20 -> 3/20
   discordantes: solo A=0 | solo B=0
   p (binomial exacto) = 1

LR suavizado  ->  Random Forest
   rutas detectadas: 3/20 -> 3/20
   discordantes: solo A=1 | solo B=1
   p (binomial exacto) = 1



**Conclusión metodológica.** A nivel ruta —la única unidad independiente que existe acá— los modelos detectan
**3 de 20 rutas de alto riesgo** y son indistinguibles entre sí.

Esto es más informativo que el resultado original. No es que el suavizado "mejoró poco": es que **el dataset no
tiene resolución suficiente para distinguir estos modelos**. Con 20 rutas en test, detectar una ruta más o una
menos mueve el recall 5 puntos. Cualquier comparación de modelos en ese rango es ruido de muestreo.

Y esto refuerza el hallazgo central del proyecto, no lo contradice: **el cuello de botella es volumen de señal,
no el algoritmo.**

## 7. El modelo de una sola variable

Segunda afirmación sin código en el repo: que entrenar sólo con `zona_riesgo_low` da casi el mismo recall que
el modelo completo. Se prueba con los dos algoritmos, porque el README no aclaraba cuál.

In [9]:
for nombre, modelo in [("Regresion logistica", LogisticRegression(max_iter=300, class_weight="balanced")),
                       ("Random Forest", RandomForestClassifier(n_estimators=100, max_depth=12,
                                                                class_weight="balanced_subsample",
                                                                random_state=42, n_jobs=-1))]:
    Xtr1, Xte1 = train_df[["zona_riesgo_low"]], test_df[["zona_riesgo_low"]]
    if nombre.startswith("Regresion"):
        sc = StandardScaler()
        Xtr1, Xte1 = sc.fit_transform(Xtr1), sc.transform(Xte1)
    modelo.fit(Xtr1, y_train)
    r = recall_score(y_test, modelo.predict(Xte1), labels=["Low"], average="macro")
    print(f"Solo zona_riesgo_low ({nombre}): recall Low = {r:.4f}")

print()
print("Referencia — modelo completo (14 features):")
print("   LR  :", round(recall_score(y_test, pred_lr, labels=["Low"], average="macro"), 4))
print("   RF  :", round(recall_score(y_test, pred_rf, labels=["Low"], average="macro"), 4))

Solo zona_riesgo_low (Regresion logistica): recall Low = 0.1992


Solo zona_riesgo_low (Random Forest): recall Low = 0.2511

Referencia — modelo completo (14 features):


   LR  : 0.1802


   RF  : 0.1772


**Discrepancia #3.** El README dice ~17%. Reproduce en **19,9%** (regresión logística) y **25,1%** (Random Forest).

El número publicado está mal, pero **la conclusión que se sacaba de él queda reforzada**: una sola variable no
sólo iguala al modelo completo, lo supera. Las otras 13 features no aportan señal; con Random Forest, agregarlas
directamente empeora la detección de `Low`.

Traducido a negocio: **el análisis por zona es el producto. El modelo multivariable, hoy, no agrega nada arriba
de eso.**

## 8. Resumen — qué se sostiene y qué no

| Afirmación del README | Reproduce | Valor real |
|---|---|---|
| Baseline LR: recall Low 18%, ROC-AUC 0.6102 | Sí, exacto | 0.180 / 0.6102 |
| Random Forest no mejora sobre el baseline | Sí | recall Low 0.177 |
| 34% de zonas con <5 rutas; zonas con tasa 100% | Sí | 33,6% / 10 zonas |
| Baseline sin suavizar = 16% | **No** | 17,95% |
| McNemar significativo, p ≈ 0.00045 | **No** | p ≈ 0.68 (parada) · p = 1 (ruta) |
| Single-feature ≈ 17% | **No** | 19,9% (LR) · 25,1% (RF) |

**Lo que cambia en el README:** la sección de validación deja de afirmar una mejora estadísticamente
significativa. Se reemplaza por el hallazgo que sí se sostiene y que es más fuerte: con 20 rutas `Low` en test,
los modelos son estadísticamente indistinguibles, y el modelo de una sola variable iguala o supera al completo.

**Lo que no cambia:** el bug del suavizado bayesiano existió y está bien corregido; el diagnóstico de zonas con
poca historia es correcto; y la conclusión de negocio del proyecto —el riesgo está concentrado geográficamente,
el modelo no es apto para uso operativo— se mantiene intacta.

**Lección de proceso, que es la que importa:** el resultado se corrió una vez en una sesión interactiva y sólo
la conclusión llegó al README. Sin el código, no había forma de detectar el error hasta que alguien intentara
reproducirlo — y ese alguien podía ser un entrevistador. Todo número publicado va con el notebook que lo produce.